# mixLSTM Test Notebook

This notebook demonstrates how to create a small sample dataset, initialize `mixLSTM`, and run a forward pass.

## 1. Environment Setup

In [1]:
import random
import numpy as np
import torch

from pyhealth.datasets import create_sample_dataset, get_dataloader
from pyhealth.datasets.splitter import split_by_sample

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running on device: {device}")

Running on device: cpu


## 2. Create Sample Dataset

Generates the synthetic per-timestep regression task from the MLHC 2019 paper.
- **Input `x`**: Sparse random matrix of shape `(T, input_dim)` with ~10% non-zero entries.
- **Target `y`**: At each timestep `t >= l` (`l = prev_used_timestamps`), `y[t]` is a weighted sum over the previous `l` timesteps and `input_dim` features. The weight distributions (`k_dist`, `d_dist`) drift slowly by `change_between_tasks` per step. For `t < l`, `y[t] = 1` (ignored during loss).

In [2]:
# Dataset parameters (matching MLHC2019 synthetic task)
num_samples = 1000
T = 30  # sequence length
input_dim = 3
prev_used_timestamps = 10
change_between_tasks = 0.05

def convert_distb(a):
        a_min = min(a)
        a_max = max(a)
        a = (a-a_min)/(a_max-a_min)
        a_sum = sum(a)
        a = a/a_sum
        return a

"""Gen X"""
x_size = num_samples*T*input_dim
x=np.zeros(x_size)
x[np.random.choice(x_size, size=int(x_size/10), replace=False)]=np.random.uniform(size=int(x_size/10))*100
x=np.resize(x, (num_samples,T,input_dim))

"""Gen y"""
k_dist = []
d_dist = []
for i in range(T):
    if i<prev_used_timestamps:
        k_dist.append(np.ones(prev_used_timestamps))
        d_dist.append(np.ones(input_dim))
    elif i==prev_used_timestamps:
        k_dist.append(convert_distb(np.random.uniform(size=(prev_used_timestamps))))
        d_dist.append(convert_distb(np.random.uniform(size=(input_dim))))
    else:
        delta_t = np.random.uniform(-change_between_tasks,change_between_tasks,size=(prev_used_timestamps))
        delta_d = np.random.uniform(-change_between_tasks,change_between_tasks,size=(input_dim))
        k_dist.append(convert_distb(k_dist[i-1]+delta_t))
        d_dist.append(convert_distb(d_dist[i-1]+delta_d))

y=np.ones((num_samples,T,1))
for i in range(T):
    if i>=prev_used_timestamps:
        y[:,i,0] = np.matmul(np.matmul(x[:,i-prev_used_timestamps:i,:],d_dist[i]), k_dist[i])

# Build samples: per-timestep regression target matching original MLHC2019 synthetic setup
samples = [
    {
        "patient_id": f"patient-{i}",
        "visit_id": "visit-0",
        "series": x[i].tolist(),
        "y": y[i].squeeze(-1).tolist(),
    }
    for i in range(num_samples)
]

input_schema = {"series": "tensor"}
output_schema = {"y": "tensor"}

dataset = create_sample_dataset(
    samples=samples,
    input_schema=input_schema,
    output_schema=output_schema,
    dataset_name="mixlstm_demo",
)

print(f"Created dataset with {len(dataset)} samples")
print(f"Input schema: {dataset.input_schema}")
print(f"Output schema: {dataset.output_schema}")

Created dataset with 1000 samples
Input schema: {'series': 'tensor'}
Output schema: {'y': 'tensor'}


## 3. Split Dataset

In [3]:
train_data, val_data, test_data = split_by_sample(dataset, [0.7, 0.15, 0.15], seed=SEED)

print(f"Train: {len(train_data)} samples")
print(f"Val: {len(val_data)} samples")
print(f"Test: {len(test_data)} samples")

train_loader = get_dataloader(train_data, batch_size=8, shuffle=True)
val_loader = get_dataloader(val_data, batch_size=8, shuffle=False)
test_loader = get_dataloader(test_data, batch_size=8, shuffle=False)

Train: 700 samples
Val: 150 samples
Test: 150 samples


## 4. Initialize `mixLSTM` Model

In [4]:
from pyhealth.models import MixLSTM

model = MixLSTM(dataset=dataset, num_experts=2, hidden_size=500,
               prev_used_timestamps=prev_used_timestamps)
model = model.to(device)
print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")
print(f"Mode: {model.mode}, Per-timestep: {model._per_timestep}")

Model created with 2021062 parameters
Mode: None, Per-timestep: True


## 5. Test Forward Pass

In [6]:
# Fetch a batch and run a forward pass
batch = next(iter(train_loader))

with torch.no_grad():
    outputs = model(**batch)

print("Output keys:", outputs.keys())
print(f"Loss (MSE): {outputs['loss'].item():.4f}")
print(f"Logit shape: {outputs['logit'].shape}")  # (batch, T, 1) for regression

Output keys: dict_keys(['logit', 'y_prob', 'loss', 'y_true'])
Loss (MSE): 61.5612
Logit shape: torch.Size([8, 30, 1])


## 6. Optional Training (Example)

The following is an example sketch for training using PyHealth's `Trainer`. Uncomment to run training.

In [7]:
from pyhealth.trainer import Trainer
trainer = Trainer(model=model, device=device)
trainer.train(
     train_dataloader=train_loader,
     val_dataloader=val_loader,
     optimizer_params={"lr": 1e-3},
     epochs=10,
     monitor="loss",
     monitor_criterion="min",
)

MixLSTM(
  (model): ExampleMowLSTM(
    (cells): ModuleList(
      (0-29): 30 x MoW(
        (experts): mowLSTM(
          (dropouts): ModuleList(
            (0): Dropout(p=0, inplace=False)
          )
          (h2o): MoO(
            (experts): ModuleList(
              (0-1): 2 x Linear(in_features=500, out_features=1, bias=True)
            )
            (gate): IDGate()
          )
          (rnns): ModuleList(
            (0): mowLSTM_(
              (input_weights): MoO(
                (experts): ModuleList(
                  (0-1): 2 x Linear(in_features=3, out_features=2000, bias=True)
                )
                (gate): IDGate()
              )
              (hidden_weights): MoO(
                (experts): ModuleList(
                  (0-1): 2 x Linear(in_features=500, out_features=2000, bias=True)
                )
                (gate): IDGate()
              )
            )
          )
        )
        (gate): NonAdaptiveGate()
      )
    )
    (experts): mow

Epoch 0 / 10:   0%|          | 0/88 [00:00<?, ?it/s]

--- Train epoch-0, step-88 ---
loss: 18.4241


Evaluation: 100%|██████████| 19/19 [00:00<00:00, 31.19it/s]

--- Eval epoch-0, step-88 ---
loss: 8.5430
New best loss score (8.5430) at epoch-0, step-88



...

--- Eval epoch-8, step-792 ---
loss: 2.0050
New best loss score (2.0050) at epoch-8, step-792


Epoch 9 / 10:   0%|          | 0/88 [00:00<?, ?it/s]

--- Train epoch-9, step-880 ---
loss: 1.7402


Evaluation: 100%|██████████| 19/19 [00:00<00:00, 33.82it/s]

--- Eval epoch-9, step-880 ---
loss: 1.9797
New best loss score (1.9797) at epoch-9, step-880
Loaded best model


## 7. Notes

- **Dataset**: Synthetic per-timestep regression task from the MLHC 2019 paper. Input `x` is a sparse random matrix `(T, input_dim)`. Target `y[t]` is a weighted combination of the previous `l` timesteps of `x`, where `l = prev_used_timestamps`. The weighting distributions (`k_dist`, `d_dist`) drift slowly over time (`change_between_tasks`).
- **Output schema**: Uses `"tensor"` (not `"regression"`) because the target is a sequence `(T,)`, not a scalar. This causes `model.mode = None` and `model._per_timestep = True`.
- **Loss**: MSE computed only on timesteps `>= prev_used_timestamps`, matching the original repo where the first `l` targets are trivially zero.
- **Classification support**: If `output_schema` is set to a standard label type (`"multiclass"`, `"binary"`, etc.), the model automatically switches to last-timestep classification using `get_loss_function()` and `prepare_y_prob()` from `BaseModel` — no flag needed.
- `MixLSTM` expects input tensors of shape `(batch, seq_len, input_dim)`.